___

# <font color= #6A994E> **Análisis de Redes Sociales - Facebook** </font>
#### <font> `PAP Movilidad Urbana Sustentable`</font>
<Strong> Oscar Josué Rocha & Viviana Toledo </Strong>

_03/11/2026._

___

# <font color= #6A994E> **Librerías & Datos** </font>

In [51]:
from pathlib import Path
import pandas as pd
import plotly.express as px

In [52]:
df = pd.read_csv('./data/clean/facebook_3months.csv')
df.head()

,url,author_name,photo_url,text,time,angry_count,hug_count,laugh_count,like_count,love_count,sad_count,surprised_count,shared_post_text,top_comment_name,top_comment_gender,top_comment_text,top2_comment_name,top2_comment_gender,top2_comment_text
0,https://www.facebook.com/groups/41352757880290...,CleverMango2834,NaN,Algun morro libre?\n🔥🍻\n\n3314873884\n\nTodo d...,2026-11-04T00:26:30.000Z,0,0,0,0,0,0,0,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,https://www.facebook.com/groups/41352757880290...,Noé Barajas,https://scontent-bos5-1.xx.fbcdn.net/v/t39.308...,¿Les ha tocado esta paradoja con los asientos?...,2026-11-04T00:10:40.000Z,1,0,10,5,0,0,0,NaN,Ana Vélez,FEMALE,"Sí, me ha tocado y te esta llendo mejor en los...",WhiteOtter8071,NaN,Ponle un clavo para el que la apartó
2,https://www.facebook.com/groups/41352757880290...,Andrea Macias,NaN,"Hola, alguien dejó esta bolsa con ropa en esta...",2026-11-03T22:20:49.000Z,0,1,0,5,0,0,1,NaN,Andrea Macias,FEMALE,Si es tuya pls mándame mensaje para regresartela,xwolfpack__222__,NaN,.
3,https://www.facebook.com/groups/41352757880290...,Bicisimio Raider,NaN,NaN,2026-11-03T02:13:17.000Z,0,0,0,0,0,0,0,"""Tengo miedo de que me pasen muy cerca los COC...",Víctor Manuel González Ramos V,MALE,Reportando desde abeldulyork.,NaN,NaN,NaN
4,https://www.facebook.com/groups/41352757880290...,Juan Carlos Gonzalez,https://scontent-iad3-1.xx.fbcdn.net/v/t39.308...,Si olvidaste tu chamarra cerca de centro médi...,2026-11-02T23:31:47.000Z,0,0,0,1,0,0,0,NaN,NaN,NaN,NaN,NaN,NaN,NaN


# <font color= #6A994E> **Análisis Exploratorio de Datos** </font>

In [53]:
# Format date
df["time"] = pd.to_datetime(df["time"], utc=True, errors="coerce")
df["time"] = df["time"].dt.tz_convert("America/Mexico_City")

# Keep only entries that have text
has_text = df["text"].fillna("").str.strip().ne("")
df_text = df[has_text]

daily = (
    df_text.set_index("time")
           .resample("D")
           .size()
           .rename("entries")
           .reset_index()
)
daily["time"] = daily["time"].dt.strftime("%Y-%m-%d")

total = int(daily["entries"].sum())

# Data
total_entries = len(df)
print(f"Pocentaje de publicaciones con texto: {(total * 100)/total_entries:.2f}%")

# Visualize data
fig = px.bar(
    daily,
    x="time",
    y="entries",
    labels={"time": "Fecha", "entries": "Publicaciones con texto"},
)
fig.update_layout(
    template="plotly_white", 
    title={
        'text' : f"Publicaciones diarias en Facebook con texto ({total:,} de {total_entries:,})", 
        'xanchor': 'center', 
        'x': 0.5,
        }, 
    bargap=0.25, 
    xaxis=dict(type="category"))
fig.show()

Pocentaje de publicaciones con texto: 75.91%


De las 631 publicaciones que descargamos a lo largo de 3 meses en el grupo de *Usuarios MiBici GDL*, 479 de ellas contienen texto, lo que representa un 76% del total. Esto es un buen acervo de comentarios, y también es una cantidad esperable, ya que no todas las publicaciones de Facebook son siempre texto. Muchas de ellas también son imágenes, publicaciones compartidas, vídeos, entre otros. 

In [67]:
# Select reaction columns
REACTION_COLS = [
    "like_count", "love_count", "hug_count", "laugh_count",
    "surprised_count", "sad_count", "angry_count",
]

# Function to count reactions
def count_reactions(df, cols=REACTION_COLS):
    counts = (
        df[cols]
        .apply(pd.to_numeric, errors="coerce")
        .fillna(0)
        .sum()
        .astype(int)
    )
    counts.index = counts.index.str.removesuffix("_count")
    counts = counts.sort_values(ascending=False)

    out = counts.rename_axis("reaction").reset_index(name="count")
    total = out["count"].sum()
    out["percent"] = out["count"] / total * 100 if total else 0.0
    out["label"] = out.apply(lambda r: f"{r['count']:,}<br>({r['percent']:.1f}%)", axis=1)
    return out

totals = count_reactions(df)

# Visualize data
fig = px.bar(
    totals,
    x="reaction",
    y="count",
    text="label",
    hover_data={"label": False, "percent": ":.1f"},
    labels={"reaction": "Reacción", "count": "Cantidad", "percent": "Porcentaje"},
)
fig.update_traces(textposition="outside", cliponaxis=False)
fig.update_yaxes(range=[0, totals["count"].max() * 1.15])  # headroom so labels aren't cut off
fig.update_layout(
    template="plotly_white",
    title={
    'text' : f"Reacciones en todas las publicaciones", 
    'xanchor': 'center', 
    'x': 0.5,
    }, 
    )
fig.show()

En las redes sociales, 'Me gusta' (like) puede ser considerada muchas veces como una reacción neutral, debido a la cualidad que tiene de expresar aprobación con una publicación. Por lo tanto, ignorando dicha reacción, que puede responder tanto a opiniones negativas o positivas, podemos observar que la reacción más común en el grupo de ***Mi Bici Usuarios GDL*** es la risa, lo cual suponemos es un indicador del humor mexicano ante la inconformidad o situaciones diversas.

Después de ello, tenemos la reacción de corazón, seguida de la reacción de tristeza, con casi la misma frecuencia.

In [ ]:
# Translate and order columns
MAP = {"FEMALE": "Mujer", "MALE": "Hombre", "NEUTER": "Neutral"}
ORDER = ["Mujer", "Hombre", "Neutral"]

pairs = [
    ("top_comment_name", "top_comment_gender"),
    ("top2_comment_name", "top2_comment_gender"),
]

# Get counts by gender
genders = pd.concat(
    [df.loc[df[name].notna(), gender] for name, gender in pairs],
    ignore_index=True,
)
# Exclude unknown genders
genders = genders.str.upper().map(MAP).dropna()  

dist = (
    genders.value_counts()
    .reindex(ORDER, fill_value=0)
    .rename_axis("gender")
    .reset_index(name="count")
)
total = dist["count"].sum()
dist["percent"] = dist["count"] / total * 100
dist["label"] = dist["percent"].map(lambda p: f"{p:.1f}%")

# Visualize data
fig = px.bar(
    dist, x="gender", y="count", text="label", color="gender",
    category_orders={"gender": ORDER},
    labels={"gender": "Género", "count": "Cantidad", "label": "Porcentaje"},
)
fig.update_traces(textposition="outside", cliponaxis=False)
fig.update_yaxes(range=[0, dist["count"].max() * 1.2])
fig.update_layout(
    template="plotly_white",
    showlegend = False,
    title={
    'text' : f"Género en los Comentarios Destacados", 
    'xanchor': 'center', 
    'x': 0.5,
    }, 
    )
fig.show()

Los datos crudos descargados de Facebook no contienen el género de la persona que realiza la publicación, pero sí contienen el género de los dos comentarios más destacados. De estas personas, excluyendo a aquellas de las que se desconoce el género, podemos observar que la mayoría de comentarios del grupo de Facebook provienen de hombres (71.5%), mientras que las mujeres y disidencias representan el (28.5%).

Estos porcentajes se alinean con la paridad de género de les usuaries de MiBici, sin embargo, podemos notar una menor participación de mujeres en el grupo de Facebook (los datos abiertos de MiBici reportan que el 34% de sus usuaries son mujeres, al 12/nov/2026). Esto puede ser indicador de una renuencia de las mujeres a comentar abiertamente en el grupo.